# Average Returns Analysis

This notebook takes inspiration from `final_report_ratings_vs_returns.ipynb`, but focuses only on the average return profile of the provided 15-stock dataset. It recomputes the horizon averages from the ticker rows and presents only aggregate outputs.

In [2]:
from __future__ import annotations

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1, palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 120,
    'figure.facecolor': 'white',
})

RETURN_DATA = [
    {'ticker': 'BRK-B', '1m': -2.4, '3m': -3.7, '6m': -2.9, '1y': -8.2, '3y': 61.4},
    {'ticker': 'CB', '1m': -2.4, '3m': 4.9, '6m': 18.2, '1y': 13.2, '3y': 82.0},
    {'ticker': 'CPRT', '1m': -6.0, '3m': -15.2, '6m': -27.0, '1y': -39.8, '3y': -7.2},
    {'ticker': 'CSU.TO', '1m': 3.4, '3m': -26.2, '6m': -40.8, '1y': -48.2, '3y': 2.8},
    {'ticker': 'EL.PA', '1m': -18.1, '3m': -28.9, '6m': -28.8, '1y': -27.9, '3y': 27.8},
    {'ticker': 'GILD', '1m': -6.6, '3m': 9.7, '6m': 22.1, '1y': 31.9, '3y': 89.8},
    {'ticker': 'MA', '1m': 0.4, '3m': -13.6, '6m': -11.9, '1y': -7.5, '3y': 44.6},
    {'ticker': 'MRSH', '1m': -3.0, '3m': -7.3, '6m': -12.5, '1y': -25.0, '3y': 12.9},
    {'ticker': 'MSFT', '1m': -3.9, '3m': -23.3, '6m': -26.4, '1y': -4.2, '3y': 36.3},
    {'ticker': 'NOVO-B.CO', '1m': -2.4, '3m': -28.3, '6m': -36.0, '1y': -53.1, '3y': -52.6},
    {'ticker': 'ROP', '1m': 2.7, '3m': -23.0, '6m': -31.2, '1y': -39.5, '3y': -17.8},
    {'ticker': 'SAP.DE', '1m': -11.4, '3m': -29.3, '6m': -35.6, '1y': -41.2, '3y': 34.0},
    {'ticker': 'V', '1m': -0.9, '3m': -14.1, '6m': -9.8, '1y': -10.8, '3y': 40.9},
    {'ticker': 'VRSK', '1m': 2.5, '3m': -10.5, '6m': -19.2, '1y': -31.0, '3y': 6.7},
    {'ticker': 'WKL.AS', '1m': 0.5, '3m': -29.2, '6m': -44.6, '1y': -55.5, '3y': -42.0},
]

HORIZONS = ['1m', '3m', '6m', '1y', '3y']
EXPECTED_AVERAGES = {'1m': -3.2, '3m': -15.9, '6m': -19.1, '1y': -23.1, '3y': 21.3}

df = pd.DataFrame(RETURN_DATA)
print(f'Dataset: {len(df)} tickers across {len(HORIZONS)} return horizons')
df.head()

Dataset: 15 tickers across 5 return horizons


## Average Returns By Horizon

In [4]:
avg_returns = (
    df[HORIZONS]
    .mean()
    .rename('Average Return (%)')
    .reset_index()
    .rename(columns={'index': 'Horizon'})
)
avg_returns['Rank'] = avg_returns['Average Return (%)'].rank(ascending=False, method='dense').astype(int)
avg_returns['Expected Average (%)'] = avg_returns['Horizon'].map(EXPECTED_AVERAGES)
avg_returns['Matches Input AVG Row'] = avg_returns.apply(
    lambda row: round(row['Average Return (%)'], 1) == round(row['Expected Average (%)'], 1),
    axis=1,
)

summary = avg_returns[['Horizon', 'Average Return (%)', 'Rank', 'Matches Input AVG Row']].copy()
summary['Average Return (%)'] = summary['Average Return (%)'].map(lambda x: f'{x:+.1f}%')
summary = summary.sort_values('Rank').reset_index(drop=True)
summary

Horizon Average Return (%)  Rank  Matches Input AVG Row
     3y             +21.3%     1                   True
     1m              -3.2%     2                   True
     3m             -15.9%     3                   True
     6m             -19.1%     4                   True
     1y             -23.1%     5                   True


## Average Return Profile

In [6]:
plot_df = avg_returns.copy()
plot_df['Color'] = plot_df['Average Return (%)'].map(lambda x: '#2ecc71' if x >= 0 else '#e74c3c')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    plot_df['Horizon'],
    plot_df['Average Return (%)'],
    color=plot_df['Color'],
    edgecolor='white',
    linewidth=0.8,
)

for bar, value in zip(bars, plot_df['Average Return (%)']):
    y = value + 1 if value >= 0 else value - 1.5
    va = 'bottom' if value >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width() / 2, y, f'{value:+.1f}%', ha='center', va=va, fontsize=9)

ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_ylabel('Average Return (%)')
ax.set_xlabel('Horizon')
ax.set_title('Average Return Across The Selected Portfolio')
plt.tight_layout()
plt.show()

Average return chart rendered successfully.


## Key Findings

In [8]:
from IPython.display import Markdown, display

best = avg_returns.loc[avg_returns['Average Return (%)'].idxmax()]
worst = avg_returns.loc[avg_returns['Average Return (%)'].idxmin()]
negative_count = int((avg_returns['Average Return (%)'] < 0).sum())
positive_count = int((avg_returns['Average Return (%)'] > 0).sum())

findings = [
    f"- The strongest average horizon is **{best['Horizon']}** at **{best['Average Return (%)']:+.1f}%**.",
    f"- The weakest average horizon is **{worst['Horizon']}** at **{worst['Average Return (%)']:+.1f}%**.",
    f"- The short-to-medium horizons are broadly weak: **{negative_count}** of **{len(avg_returns)}** horizons are negative.",
    f"- Only **{positive_count}** horizon is positive on average, which suggests the basket looks much better over **3y** than over the more recent windows.",
]

display(Markdown('\n'.join(findings)))

- The strongest average horizon is **3y** at **+21.3%**.
- The weakest average horizon is **1y** at **-23.1%**.
- The short-to-medium horizons are broadly weak: **4** of **5** horizons are negative.
- Only **1** horizon is positive on average, which suggests the basket looks much better over **3y** than over the more recent windows.
